# 08 — Fine-tune the Traffic-Light Detector on Your Videos

Adapt the Stage-1 detector to **your** dashcam footage (the videos in
`data/raw/traffic/video/`) **without losing the existing 26k-image dataset**.

## Why
The current `detector_model.pt` was trained on LISA (US traffic lights) and only
*weakly* recognises the lights in your videos (confidence 0.29–0.6, flickery).
Fine-tuning on corrected frames from your own footage raises confidence and makes
detection **stable frame-to-frame**.

## What this does (and does NOT change)
- **Adds** corrected video frames to `detector_yolo/` — existing images untouched.
- **Continues training** from `detector_model.pt` (keeps pothole + LISA knowledge).
- Leaves `detector_dashcam.pt`, `severity_model.pt`, and all inference code as-is.

## The one manual step
The COCO + your-model union only *proposes* boxes (~70–80% right). **You must
correct them** (Step 3) — delete the black-rectangle false positives, add missed
lights. That correction is what actually fixes the model.

## Step 0 — Setup & paths

In [ ]:
import sys, subprocess
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
print("Project root:", BASE_DIR)

VIDEO_DIR    = BASE_DIR / "data" / "raw" / "traffic" / "video"
FRAMES_DIR   = BASE_DIR / "data" / "raw" / "traffic" / "video_frames"
LABELED_DIR  = BASE_DIR / "data" / "raw" / "traffic" / "video_labeled"
DATASET_YAML = BASE_DIR / "data" / "processed" / "detector_yolo" / "dataset.yaml"
MODEL_PATH   = BASE_DIR / "models" / "detector_model.pt"

print("Videos found:", [p.name for p in sorted(VIDEO_DIR.glob('*.mp4'))])

## Step 1 — Extract frames + auto-pre-label

Runs `src/prepare_traffic_video.py`: pulls sharp, de-duplicated frames from every
video, then proposes traffic-light boxes using the **union of COCO YOLOv8s
(class 9) and your current model** (they catch different lights). Output is staged
in `data/raw/traffic/video_labeled/`.

> The 621 MB *traffic light hard.mp4* is your most valuable source — it takes the
> longest to read but gives the hardest, most useful frames. Lower `--every` to
> sample more densely, or point `--source` at a single video to start small.

In [ ]:
EVERY          = 1.0    # seconds between sampled frames
MAX_PER_VIDEO  = 400
CONF           = 0.25   # recall-first: better to over-propose and delete

subprocess.run([
    sys.executable, str(BASE_DIR / "src" / "prepare_traffic_video.py"),
    "--every", str(EVERY), "--max-per-video", str(MAX_PER_VIDEO), "--conf", str(CONF),
], check=True)

n = len(list((LABELED_DIR / "images").glob("*.jpg")))
print(f"\n{n} pseudo-labelled frames staged → {LABELED_DIR}")

## Step 2 — Preview the proposed labels (eyeball before correcting)

Green boxes are the auto-proposals. Scan them: note the **wrong** ones (black
rectangles, signs) you'll delete and the **missed** lights you'll add in Step 3.

In [ ]:
def draw_boxes(img, label_path):
    h, w = img.shape[:2]
    if label_path.exists():
        for ln in label_path.read_text().splitlines():
            p = ln.split()
            if len(p) != 5:
                continue
            cx, cy, bw, bh = map(float, p[1:])
            x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
            x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return img

imgs   = sorted((LABELED_DIR / "images").glob("*.jpg"))
sample = imgs[:9]
if sample:
    fig, axes = plt.subplots(3, 3, figsize=(16, 9))
    for ax, ip in zip(axes.ravel(), sample):
        im = draw_boxes(cv2.imread(str(ip)), LABELED_DIR / "labels" / (ip.stem + ".txt"))
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(ip.name[:24], fontsize=8)
    plt.tight_layout(); plt.show()
    print("Green = proposals. Wrong ones → delete in Step 3. Missed lights → add.")
else:
    print("No staged frames — run Step 1 first.")

## Step 3 — ✋ Correct the labels with makesense.ai (free, browser)

No install, no account, no credits. Work in **batches of ~100–150 images** (the UHD
frames are heavy to render in-browser).

1. Open <https://www.makesense.ai> → **Get Started**.
2. **Drop the images** from `data/raw/traffic/video_labeled/images/` (one batch).
3. Choose **Object Detection**.
4. **Define labels in this exact order** (the index matters): `pothole`, then
   `traffic_light`. Easiest: click *Load labels from file* and pick
   `data/raw/traffic/video_labeled/labels/classes.txt`.
5. **Actions → Import Annotations → YOLO**, select the per-image `.txt` files from
   `labels/` (skip `classes.txt`). Your green proposal boxes appear on each frame.
6. **Correct**: delete the wrong boxes (the black-rectangle FPs), draw any missed
   lights with the `traffic_light` label selected.
7. **Actions → Export Annotations → YOLO (.zip) → Export.**
8. Unzip and copy the per-image `.txt` files back into
   `data/raw/traffic/video_labeled/labels/`, overwriting. **Keep `classes.txt`.**

> ⚠️ Step 4's label order must be `pothole`(0), `traffic_light`(1) — otherwise the
> class indices won't match the dataset. Loading `classes.txt` guarantees it.

Then run **Step 3b** below, and continue to Step 4.

*Prefer a desktop app? `pip install labelImg` works too, or self-host CVAT — both free.*

In [ ]:
# Step 3b — finalize: make sure every staged frame has a label file.
# makesense exports a .txt only for frames that still have boxes. Frames where you
# deleted every false box should become BACKGROUND (hard negatives) — give them an
# empty .txt so the merge KEEPS them (teaches "these shapes are not lights") instead
# of dropping them.
img_dir = LABELED_DIR / "images"
lbl_dir = LABELED_DIR / "labels"
made = 0
for ip in img_dir.glob("*.jpg"):
    lp = lbl_dir / (ip.stem + ".txt")
    if not lp.exists():
        lp.write_text("")          # background frame
        made += 1
total = len(list(img_dir.glob("*.jpg")))
print(f"Created {made} empty (background) labels. {total} frames ready for merge.")

## Step 4 — Merge into the existing dataset (additive)

Runs `src/merge_traffic_video.py`: copies your corrected frames into
`detector_yolo/{train,val}` with a `vid_` prefix (existing images untouched),
oversampling each new train frame so a few hundred frames aren't drowned by 26k.

In [ ]:
OVERSAMPLE = 8   # copies of each new train frame (lower to 4 if disk is tight)

subprocess.run([
    sys.executable, str(BASE_DIR / "src" / "merge_traffic_video.py"),
    "--oversample", str(OVERSAMPLE),
], check=True)

## Step 5 — Fine-tune

Continues from `detector_model.pt` with a **low learning rate** and `freeze=10`
(freezes the backbone) so it adapts to your lights without forgetting potholes or
LISA. `imgsz=1280` because traffic lights are small.

> **GPU strongly recommended** — fine-tuning over the full dataset on CPU is very
> slow. For a quick smoke test, set `EPOCHS=3` and `FRACTION=0.2`.

In [ ]:
import torch
from ultralytics import YOLO

EPOCHS   = 30
IMGSZ    = 1280
FRACTION = 1.0   # fraction of the dataset per epoch (lower = faster, less thorough)
device   = "0" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

model = YOLO(str(MODEL_PATH))
model.train(
    data      = str(DATASET_YAML),
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    fraction  = FRACTION,
    lr0       = 0.001,
    freeze    = 10,
    batch     = -1,
    optimizer = "AdamW",
    cos_lr    = True,
    patience  = 15,
    project   = str(BASE_DIR / "runs"),
    name      = "detector_traffic_ft",
    exist_ok  = True,
    device    = device,
    plots     = True,
)
best = BASE_DIR / "runs" / "detector_traffic_ft" / "weights" / "best.pt"
print("\nbest.pt:", best, "exists:", best.exists())

## Step 5b — Promote the fine-tuned model (keeps a backup)

Backs up the current model to `detector_model_prefinetune.pt`, then swaps in the
fine-tuned weights so the app + pipeline use them. Skip this cell if Step 6 shows
the fine-tune did **not** improve traffic-light AP.

In [ ]:
import shutil
best   = BASE_DIR / "runs" / "detector_traffic_ft" / "weights" / "best.pt"
target = BASE_DIR / "models" / "detector_model.pt"

if best.exists():
    shutil.copy2(target, BASE_DIR / "models" / "detector_model_prefinetune.pt")
    shutil.copy2(best, target)
    print("Promoted fine-tuned model → models/detector_model.pt")
    print("Backup saved → models/detector_model_prefinetune.pt")
else:
    print("No best.pt found — training may not have finished.")

## Step 6 — Evaluate: traffic-light AP before vs after

Validates the **pre-finetune backup** and the **fine-tuned** model on the same val
split and prints per-class numbers. You want `traffic_light` AP@0.5 to go **up**.

In [ ]:
from ultralytics import YOLO

def per_class(weights):
    m = YOLO(str(weights))
    r = m.val(data=str(DATASET_YAML), split="val", verbose=False)
    b = r.box
    return {m.names[int(ci)]: b.class_result(i)[2]   # AP@0.5
            for i, ci in enumerate(b.ap_class_index)}

pre_path = BASE_DIR / "models" / "detector_model_prefinetune.pt"
print("AP@0.5 per class (val):")
if pre_path.exists():
    print("  BEFORE:", {k: round(v, 3) for k, v in per_class(pre_path).items()})
print("  AFTER :", {k: round(v, 3) for k, v in per_class(MODEL_PATH).items()})

## Step 7 — Use it

The pipeline already loads `models/detector_model.pt`, so just restart the backend:

```bash
python backend/app.py
```

Re-run your test video. Combined with the inference fixes already in place
(false-positive rejection, single lane-relevant light, ByteTrack tracking, the
black-rectangle colour fix), traffic-light detection should now be **stable, with
one clear light per lane and far fewer false positives**.

### If traffic AP didn't improve
- Correct **more** frames (especially from *traffic light hard.mp4*).
- Raise `OVERSAMPLE` (Step 4) so the new frames carry more weight.
- Train longer / on a GPU.